In [1]:
import os
import smtplib
import requests
import wikipedia
 
from email.mime.multipart import MIMEMultipart
from email.mime.text      import MIMEText
from dotenv               import load_dotenv
from langchain_openai     import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools    import tool
from langgraph.prebuilt      import create_react_agent

In [2]:
# CONFIGURACIÓN

load_dotenv()  # Lee el archivo .env automáticamente
 
OPENAI_API_KEY      = os.getenv("OPENAI_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
 
# Credenciales Gmail — agrégalas en tu .env
GMAIL_REMITENTE     = os.getenv("GMAIL_REMITENTE")      # ej: agencia@gmail.com
GMAIL_APP_PASSWORD  = os.getenv("GMAIL_APP_PASSWORD")   # contraseña de aplicación Gmail
 
# Debug: verifica que las keys se carguen (elimina estas líneas en producción)
print(f"[DEBUG] OPENWEATHER_API_KEY: {'✅ cargada' if OPENWEATHER_API_KEY else '❌ NO encontrada'}")
print(f"[DEBUG] GMAIL_REMITENTE    : {'✅ cargada' if GMAIL_REMITENTE    else '❌ NO encontrada'}")
 
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.7,
    streaming=True,
)

[DEBUG] OPENWEATHER_API_KEY: ✅ cargada
[DEBUG] GMAIL_REMITENTE    : ✅ cargada


In [3]:
TRANSPORTE = {
    "puerto montt": {
        "auto":  {"duracion": "Destino local", "peajes": "$0 CLP"},
        "bus":   {"duracion": "Destino local", "precio": "$0 CLP"},
        "ferry": None,
    },
    "puerto varas": {
        "auto":  {"duracion": "30 min",  "peajes": "$0 CLP"},
        "bus":   {"duracion": "40 min",  "precio": "$1.500 CLP"},
        "ferry": None,
    },
    "chiloe": {
        "auto":  {"duracion": "2h 30min (incluye transbordador)", "peajes": "$4.500 CLP"},
        "bus":   {"duracion": "3h",  "precio": "$6.000 CLP"},
        "ferry": {"ruta": "Pargua → Chacao", "duracion": "30 min", "precio": "$4.500 CLP (auto)"},
    },
    "frutillar": {
        "auto":  {"duracion": "1h",       "peajes": "$0 CLP"},
        "bus":   {"duracion": "1h 20min", "precio": "$2.500 CLP"},
        "ferry": None,
    },
}
 
# ──────────────────────────────────────────────
# HERRAMIENTAS (TOOLS)
# ──────────────────────────────────────────────
 
@tool
def paquetes_turisticos(zona: str) -> str:
    """
    Entrega los paquetes turísticos disponibles para una zona.
    Zonas válidas: puerto montt, puerto varas, chiloe, frutillar.
    """
    zona = zona.lower()
 
    paquetes = {
        "puerto montt": [
            "Tour Angelmó + Costanera (medio día)",
            "Tour Alerce Andino (día completo)",
            "Tour Reloncaví (medio día)",
        ],
        "puerto varas": [
            "Tour Lago Llanquihue (medio día)",
            "Tour Saltos del Petrohué (día completo)",
            "Tour Volcán Osorno (día completo)",
        ],
        "chiloe": [
            "Tour Castro (medio día)",
            "Tour Iglesias Patrimoniales (día completo)",
            "Tour Pingüineras (día completo)",
        ],
        "frutillar": [
            "Tour Teatro del Lago (medio día)",
            "Tour Costanera de Frutillar (medio día)",
            "Tour Cultural Alemán (medio día)",
        ],
    }
 
    if zona in paquetes:
        respuesta = f"Paquetes disponibles en {zona.title()}:\n"
        for p in paquetes[zona]:
            respuesta += f"  - {p}\n"
        return respuesta
 
    return f"No hay paquetes registrados para '{zona}'. Zonas disponibles: {', '.join(paquetes.keys())}."
 
 
@tool
def precio_tour(personas: int) -> str:
    """
    Calcula el precio total de un tour según la cantidad de personas.
    """
    valor_persona = 50_000
    total         = personas * valor_persona
    return f"El precio total para {personas} persona(s) es ${total:,} pesos chilenos."
 
 
@tool
def informacion_3m_tours(texto: str) -> str:
    """
    Entrega información general sobre la empresa 3M Tours.
    """
    return (
        "3M Tours es una agencia de turismo ubicada en Puerto Montt. "
        "Ofrece tours y paquetes turísticos en la Región de Los Lagos. "
        "Especialistas en recorridos naturales, volcanes, lagos y turismo aventura."
    )
 
 
@tool
def clima_zona(zona: str) -> str:
    """
    Consulta el clima actual de una zona turística usando OpenWeatherMap.
    Zonas válidas: puerto montt, puerto varas, chiloe, frutillar.
    """
    ciudades = {
        "puerto montt": "Puerto Montt,CL",
        "puerto varas": "Puerto Varas,CL",
        "chiloe":       "Castro,CL",
        "frutillar":    "Frutillar,CL",
    }
 
    zona_key = zona.lower().strip()
    ciudad   = ciudades.get(zona_key)
 
    if not ciudad:
        return (
            f"No se puede consultar el clima para '{zona}'. "
            f"Zonas válidas: {', '.join(ciudades.keys())}."
        )
 
    if not OPENWEATHER_API_KEY:
        return "❌ Error: OPENWEATHER_API_KEY no está configurada en el archivo .env"
 
    try:
        url    = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q":     ciudad,
            "appid": OPENWEATHER_API_KEY,
            "units": "metric",
            "lang":  "es",
        }
        resp = requests.get(url, params=params, timeout=8)
 
        # Si hay error HTTP, mostramos el detalle exacto
        if resp.status_code != 200:
            return (
                f"❌ Error al consultar el clima (HTTP {resp.status_code}): "
                f"{resp.json().get('message', 'sin detalle')}"
            )
 
        data        = resp.json()
        descripcion = data["weather"][0]["description"].capitalize()
        temp        = data["main"]["temp"]
        sensacion   = data["main"]["feels_like"]
        humedad     = data["main"]["humidity"]
        viento_kmh  = round(data["wind"]["speed"] * 3.6, 1)
 
        lluvia = "rain" in data or any(
            w in descripcion.lower() for w in ["lluvia", "llovizna", "chubascos", "tormenta"]
        )
 
        if lluvia:
            condicion = "⚠️ Hay lluvia — se recomienda tours cubiertos o de interior."
        elif temp < 8:
            condicion = "🥶 Hace bastante frío — abrígate bien para tours al aire libre."
        elif temp >= 18:
            condicion = "☀️ Día agradable — ideal para tours al aire libre."
        else:
            condicion = "🌤️ Clima fresco pero aceptable para la mayoría de los tours."
 
        return (
            f"Clima actual en {zona.title()}:\n"
            f"  - Descripción : {descripcion}\n"
            f"  - Temperatura : {temp}°C (sensación {sensacion}°C)\n"
            f"  - Humedad     : {humedad}%\n"
            f"  - Viento      : {viento_kmh} km/h\n"
            f"  - Evaluación  : {condicion}"
        )
 
    except requests.RequestException as e:
        return f"❌ Error de red al consultar el clima: {e}"
 
 
@tool
def opciones_transporte(zona: str) -> str:
    """
    Devuelve las opciones de transporte desde Puerto Montt hacia la zona indicada,
    incluyendo auto (duración + peajes), bus (duración + precio) y ferry si aplica.
    """
    zona_key = zona.lower().strip()
    info     = TRANSPORTE.get(zona_key)
 
    if not info:
        return f"No hay información de transporte para '{zona}'."
 
    resultado = f"Opciones de transporte hacia {zona.title()} (desde Puerto Montt):\n"
 
    auto = info.get("auto")
    if auto:
        resultado += f"  🚗 Auto  : {auto['duracion']} — Peajes: {auto['peajes']}\n"
 
    bus = info.get("bus")
    if bus:
        resultado += f"  🚌 Bus   : {bus['duracion']} — Precio: {bus['precio']}\n"
 
    ferry = info.get("ferry")
    if ferry:
        resultado += (
            f"  ⛴️  Ferry : {ferry['ruta']} — {ferry['duracion']} "
            f"— Precio: {ferry['precio']}\n"
        )
 
    return resultado
 
 
@tool
def enviar_confirmacion_email(
    email_cliente: str,
    nombre_cliente: str,
    tours: str,
    fecha: str,
    personas: int,
    zona: str,
    transporte_elegido: str,
) -> str:
    """
    Envía un correo de confirmación al cliente con el resumen de su itinerario.
 
    Parámetros:
    - email_cliente     : correo del cliente (ej: juan@gmail.com)
    - nombre_cliente    : nombre del cliente
    - tours             : tours seleccionados (texto libre)
    - fecha             : fecha del tour (ej: 25 de mayo de 2025)
    - personas          : cantidad de personas
    - zona              : zona o destino elegido
    - transporte_elegido: medio de transporte elegido (auto, bus o ferry)
    """
    if not GMAIL_REMITENTE or not GMAIL_APP_PASSWORD:
        return (
            "❌ Error: Las credenciales de Gmail no están configuradas en el .env.\n"
            "Agrega GMAIL_REMITENTE y GMAIL_APP_PASSWORD."
        )
 
    valor_persona = 50_000
    precio_total  = personas * valor_persona
 
    # ── Cuerpo HTML del correo ──────────────────
    html = f"""
    <html><body style="font-family:Arial,sans-serif; color:#333; max-width:600px; margin:auto;">
 
      <div style="background:#1a6b3c; padding:20px; border-radius:8px 8px 0 0; text-align:center;">
        <h1 style="color:white; margin:0;">🌿 3M Tours</h1>
        <p style="color:#c8f0d8; margin:4px 0;">Confirmación de Itinerario</p>
      </div>
 
      <div style="background:#f9f9f9; padding:24px; border:1px solid #ddd; border-top:none; border-radius:0 0 8px 8px;">
 
        <p>Hola <strong>{nombre_cliente}</strong>, ¡gracias por elegir 3M Tours! 🎉</p>
        <p>Tu itinerario ha sido agendado exitosamente. Aquí tienes el resumen:</p>
 
        <table style="width:100%; border-collapse:collapse; margin:16px 0;">
          <tr style="background:#e8f5ee;">
            <td style="padding:10px; border:1px solid #c3e6cb; font-weight:bold;">📍 Destino</td>
            <td style="padding:10px; border:1px solid #c3e6cb;">{zona.title()}</td>
          </tr>
          <tr>
            <td style="padding:10px; border:1px solid #ddd; font-weight:bold;">🗓️ Fecha</td>
            <td style="padding:10px; border:1px solid #ddd;">{fecha}</td>
          </tr>
          <tr style="background:#e8f5ee;">
            <td style="padding:10px; border:1px solid #c3e6cb; font-weight:bold;">🎒 Tours</td>
            <td style="padding:10px; border:1px solid #c3e6cb;">{tours}</td>
          </tr>
          <tr>
            <td style="padding:10px; border:1px solid #ddd; font-weight:bold;">👥 Personas</td>
            <td style="padding:10px; border:1px solid #ddd;">{personas}</td>
          </tr>
          <tr style="background:#e8f5ee;">
            <td style="padding:10px; border:1px solid #c3e6cb; font-weight:bold;">🚌 Transporte</td>
            <td style="padding:10px; border:1px solid #c3e6cb;">{transporte_elegido.title()}</td>
          </tr>
          <tr>
            <td style="padding:10px; border:1px solid #ddd; font-weight:bold;">💰 Precio total</td>
            <td style="padding:10px; border:1px solid #ddd;"><strong>${precio_total:,} CLP</strong></td>
          </tr>
        </table>
 
        <p style="background:#fff8e1; padding:12px; border-left:4px solid #f9a825; border-radius:4px;">
          ⏰ <strong>Recuerda:</strong> presentarte 15 minutos antes en el punto de encuentro.
          Ante cualquier duda, contáctanos respondiendo este correo.
        </p>
 
        <p style="text-align:center; margin-top:24px; color:#555;">
          ¡Te esperamos para vivir una experiencia única en la Región de Los Lagos! 🏔️🌊
        </p>
 
      </div>
 
      <p style="text-align:center; font-size:12px; color:#aaa; margin-top:12px;">
        3M Tours — Puerto Montt, Chile
      </p>
 
    </body></html>
    """
 
    # ── Envío por SMTP ──────────────────────────
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"✅ Confirmación de tu tour en {zona.title()} — 3M Tours"
        msg["From"]    = GMAIL_REMITENTE
        msg["To"]      = email_cliente
        msg.attach(MIMEText(html, "html"))
 
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(GMAIL_REMITENTE, GMAIL_APP_PASSWORD)
            server.sendmail(GMAIL_REMITENTE, email_cliente, msg.as_string())
 
        return (
            f"✅ Correo de confirmación enviado exitosamente a {email_cliente}.\n"
            f"Resumen: {personas} persona(s) — {zona.title()} — {fecha} — ${precio_total:,} CLP."
        )
 
    except smtplib.SMTPAuthenticationError:
        return (
            "❌ Error de autenticación Gmail. Verifica que:\n"
            "1. GMAIL_REMITENTE sea tu correo correcto.\n"
            "2. GMAIL_APP_PASSWORD sea una contraseña de aplicación (no tu clave normal).\n"
            "   Créala en: myaccount.google.com → Seguridad → Contraseñas de aplicación."
        )
    except Exception as e:
        return f"❌ Error al enviar el correo: {e}"
 

In [4]:
# AGENTE
# ──────────────────────────────────────────────
 
tools = [
    paquetes_turisticos,
    precio_tour,
    informacion_3m_tours,
    clima_zona,
    opciones_transporte,
    enviar_confirmacion_email,
]
 
SYSTEM_PROMPT = """
Eres un asistente virtual de 3M Tours, una agencia de turismo de Puerto Montt.
 
Tu misión es ayudar al cliente a planificar y confirmar un itinerario turístico personalizado.
 
Cuando el cliente mencione un destino o quiera visitar alguna zona, debes:
1. Consultar el clima actual con `clima_zona`.
2. Listar los paquetes disponibles con `paquetes_turisticos`.
3. Mostrar las opciones de transporte con `opciones_transporte`.
4. Considerar el tiempo disponible que el cliente mencione (ej: "tengo 2 días").
5. Armar un itinerario claro, ordenado por día si aplica, adaptado al clima.
 
Cuando el cliente confirme su tour, debes pedirle:
- Su nombre completo
- Su correo electrónico
- La fecha en que quiere realizar el tour
- El medio de transporte elegido (auto, bus o ferry)
 
Con esos datos, usa `enviar_confirmacion_email` para enviar la confirmación.
 
Reglas:
- Si el clima es malo (lluvia, viento fuerte), sugiere tours cubiertos o de interior.
- Si el clima es bueno, prioriza tours al aire libre.
- Si el cliente pregunta por precios, usa `precio_tour`.
- Responde siempre en español, de forma amigable y profesional.
"""
 
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)
 
# ──────────────────────────────────────────────
# CHAT LOOP
# ──────────────────────────────────────────────
 
print("------------------------------------")
print("      BIENVENIDO A 3M TOURS")
print("------------------------------------")
print("(escribe 'salir' para terminar)\n")
 
conversation_history = []
 
while True:
    pregunta = input("Cliente: ").strip()
 
    if not pregunta:
        continue
 
    if pregunta.lower() == "salir":
        print("Gracias por preferir 3M Tours. ¡Buen viaje!")
        break
 
    conversation_history.append(HumanMessage(content=pregunta))
 
    respuesta = agent.invoke({"messages": conversation_history})
 
    messages    = respuesta.get("messages", [])
    agente_text = messages[-1].content if messages else str(respuesta)
 
    # Mantener historial para contexto multi-turno
    conversation_history = messages
 
    print(f"\nAgente:\n{agente_text}\n")
    print("------------------------------------")

------------------------------------
      BIENVENIDO A 3M TOURS
------------------------------------
(escribe 'salir' para terminar)



C:\Users\matis\AppData\Local\Temp\ipykernel_9356\2564166466.py:40: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



Agente:
¡Hola Matías! Aquí tienes el itinerario que he preparado para ti y Renato para los días 23 y 24 de mayo, considerando que el clima en Puerto Montt es cielo claro, pero hace bastante frío, así que es importante abrigarse bien:

### Itinerario Propuesto

#### Día 1: 23 de mayo
- **Tour Angelmó + Costanera (medio día)**: Inicia tu día explorando el famoso mercado de Angelmó, donde podrás disfrutar de la gastronomía local y luego dar un paseo por la costanera, disfrutando de las vistas del puerto. 
  - **Duración**: Medio día

#### Día 2: 24 de mayo
- **Tour Alerce Andino (día completo)**: Este es un hermoso parque nacional donde podrás disfrutar de la naturaleza, realizar caminatas y observar la flora y fauna local. Es un tour al aire libre, por lo que asegúrate de llevar abrigo.
  - **Duración**: Día completo

### Opciones de Transporte
- **Auto**: Destino local — Peajes: $0 CLP
- **Bus**: Destino local — Precio: $0 CLP

### Precio Total
- El precio total para 2 personas es **$1